In [49]:
import torch
import pandas as pd
import torch.nn.functional as F

# load embedding
data = torch.load("data/article_embeddings2.pt")
emb = data["embeddings"]
id2idx = data["id2idx"]

# normalize (QUAN TRỌNG)
emb = F.normalize(emb, dim=1)

# load split
train_df = pd.read_csv("data/train_split.csv")
test_df = pd.read_csv("data/test_split.csv")

# đảm bảo format giống nhau
train_df["article_id"] = train_df["article_id"].astype(str).str.zfill(10)
test_df["article_id"] = test_df["article_id"].astype(str).str.zfill(10)

print("Train:", len(train_df))
print("Test:", len(test_df))

Train: 216048
Test: 29288


In [50]:
from collections import defaultdict

user_hist = defaultdict(list)

for _, row in train_df.iterrows():
    user_hist[row["customer_id"]].append(row["article_id"])

print("Num users:", len(user_hist))

Num users: 54987


In [51]:
idx2id = {v: k for k, v in id2idx.items()}

In [52]:
def recommend(user_items, emb, id2idx, idx2id, K=10):
    # 🔥 chỉ lấy last N items
    user_items = user_items[-5:]   # QUAN TRỌNG

    vecs = []
    for item in user_items:
        if item in id2idx:
            vecs.append(emb[id2idx[item]])

    if len(vecs) == 0:
        return []

    user_emb = torch.stack(vecs).mean(dim=0)

    scores = torch.matmul(emb, user_emb)

    # remove seen
    seen = set(user_items)

    topk = torch.topk(scores, k=K + len(seen)).indices

    results = []
    for i in topk:
        aid = idx2id[i.item()]
        if aid not in seen:
            results.append(aid)
        if len(results) == K:
            break

    return results

In [53]:
import math

def evaluate(test_df, user_hist, emb, id2idx, idx2id, K=10):
    hits = 0
    ndcg = 0
    total = 0

    for _, row in test_df.iterrows():
        user = row["customer_id"]
        true_item = row["article_id"]

        if user not in user_hist:
            continue

        preds = recommend(user_hist[user], emb, id2idx, idx2id, K)

        if len(preds) == 0:
            continue

        # Recall / Hit
        if true_item in preds:
            hits += 1

            rank = preds.index(true_item)
            ndcg += 1 / math.log2(rank + 2)

        total += 1

    return {
        f"Recall@{K}": hits / total,
        f"NDCG@{K}": ndcg / total
    }

In [54]:
for k in [5, 10, 20]:
    result = evaluate(test_df, user_hist, emb, id2idx, idx2id, K=k)
    print(result)

{'Recall@5': 0.01903303434697003, 'NDCG@5': 0.012321540913470189}
{'Recall@10': 0.03565959308685189, 'NDCG@10': 0.017617823766620775}
{'Recall@20': 0.05031721723911617, 'NDCG@20': 0.02131522396719154}


In [55]:
missing = sum(1 for _, row in test_df.iterrows() 
              if row["article_id"] not in id2idx)

print("Missing ratio:", missing / len(test_df))

Missing ratio: 0.04479650368751707
